# DAY 9

In [ ]:
import pdfplumber
from openai import OpenAI
import chromadb

pdf_path = "ml_notes.pdf"

with open("api_key.txt","r") as f:
    api_key = f.read().strip()

**0. Clean Text**

In [ ]:
import re

def clean_text(text: str):
    # 1. Remove bullet symbols & special chars
    text = re.sub(r'[\u2022\u2023\u25E6\u2043\u2219\uf0b7]', ' ', text)

    # 2. Remove page numbers (lines with only digits)
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)

    # 3. Remove repeated headers/footers (heuristic)
    text = re.sub(r'(Page\s*\d+)|(Lecture\s*\d+)', '', text, flags=re.IGNORECASE)

    # 4. Fix hyphenated line breaks
    text = re.sub(r'-\n', '', text)

    # 5. Merge broken lines inside paragraphs
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)

    # 6. Normalize whitespace
    text = re.sub(r'\n{2,}', '\n\n', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

**1. Ingestion: load + chunk**

In [ ]:
from pypdf import PdfReader

def load_and_chunk_pdf(pdf_path: str, chunk_size: int = 2500, overlap: int = 500):

    """ 
    Loads a PDF, extract text, chunk it using 
    fixed-size overlapping windows.
    """
    reader = PdfReader(pdf_path)

    raw_text = ""
    for page in reader.pages:
        raw_text += page.extract_text() + "\n"

    cleaned_text = clean_text(raw_text)

    chunks = []
    start = 0

    while start < len(cleaned_text):
        end = start + chunk_size
        chunk = cleaned_text[start:end]
        chunks.append(chunk)
        start = end - overlap

    return chunks

In [ ]:
chunks = load_and_chunk_pdf("ml_notes.pdf")

print(chunks[:300])

['1 UNIT I Introduction to Machine Learning 1. Introduction 1.1 What Is Machine Learning? Machine learning is programming computers to optimize a performance criterion using example data or past experience. We have a model defined up to some parameters, and learning is the execution of a computer program to optimize the parameters of the model using the training data or past experience. The model may be predictive to make predictions in the future, or descriptive to gain knowledge from data, or both. Arthur Samuel, an early American leader in the field of computer gaming and artificial intellige nce, coined the term “Machine Learning” in 1959 while at IBM. He defined machine learning as “the field of study that gives computers the ability to learn without being explicitly programmed.” However, there is no universally accepted definition for machine learning. Different authors define the term differently. Definition of learning Definition A computer program is said to learn from experie

**2. Embedding helpers (single source of truth)**

In [ ]:
client = OpenAI(api_key= api_key)

def embed_documents(texts):
    return [
        client.embeddings.create(
            model= "text-embedding-3-small",
            input= text
        ).data[0].embedding
        for text in texts
    ]

def embed_query(query):
    return client.embeddings.create(
        model= "text-embedding-3-small",
        input= query
    ).data[0].embedding

**3. Ingestion: build vector store (RUN ONCE)**

In [ ]:
def build_vector_store(chunks):
    client = chromadb.PersistentClient(path="./chromaDB_store")
    collection = client.get_or_create_collection(name="ml_notes")

    embeddings= embed_documents(chunks)

    collection.add(
        ids= [str(i) for i in range(len(chunks))],
        documents= chunks,
        embeddings= embeddings 
    )

**4. Retrieval: load store**

In [ ]:
def load_vector_store():
    client = chromadb.PersistentClient(path="./chromaDB_store")
    return client.get_collection("ml_notes")

**5. Retrieval: top-k search**

In [ ]:
def retrive_top_k(query, k =3):
    collection = load_vector_store()

    query_embeddings = embed_query(query)

    results = collection.query(
        query_embeddings=[query_embeddings],
        n_results= k
    )

    return results["documents"][0]

In [ ]:
chunks = load_and_chunk_pdf("ml_notes.pdf")
build_vector_store(chunks)

In [ ]:
retrive_top_k("Explain Candidate Elimination algorithm")

['of VSH,D. 2. It can be proven by assuming some h in VSH,D,that does n ot satisfy the right -hand side of the expression, then showing that this leads to an inconsistency 1.7.3 CANDIDATE-ELIMINATION Learning Algorithm The CANDIDATE-ELIMINTION algorithm computes the version space containing all hypotheses from H that are consistent with an observed sequence of training examples. Initialize G to the set of maximally general hypotheses in H Initialize S to the set of maximally specific hypotheses in H For each training example d, do If d is a positive example Remove from G any hypothesis inconsistent with d For each hypothesis s in S that is not consistent with d Remove s from S Add to S all minimal generalizations h of s such that h is consistent with d, and some member of G is more general than h Remove from S any hypothesis that is more general than another hypothesis in S If d is a negative example Remove from S any hypothesis inconsistent with d For each hypothesis g in G that is no